# Libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
from plotnine import *

import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
pd.set_option("display.max_columns", None)

import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import shap
import ast  # Helps convert string representations of lists

In [2]:
!git add .
!git commit -m "chained stuff together"
!git push

[main d9445bd] chained stuff together
 1 file changed, 927 insertions(+), 189 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 64 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 7.50 KiB | 767.00 KiB/s, done.
Total 4 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To github.com:innacohen/mod-extract.git
   d8fb692..d9445bd  main -> main


# Functions

In [3]:
def plot_countplot(
    df,
    col,
    subset_query=None,
    title=None,
    color='#4C72B0',
    horizontal=True,
    figsize=(8, 5),
    show_counts=False
):
    """
    Create a countplot from a column in a pandas DataFrame.

    Parameters:
    -----------
    df : pandas.DataFrame
        Your input DataFrame.
    col : str
        Column to count and plot.
    subset_query : str, optional
        Optional query string to filter the DataFrame (e.g., "type == 'Neither'").
    title : str, optional
        Title for the plot.
    color : str, optional
        Bar color (default: '#4C72B0').
    horizontal : bool, default True
        If True, plot bars horizontally.
    figsize : tuple, optional
        Figure size in inches.
    show_counts : bool, default False
        If True, annotate bars with counts.
    """
    data = df.query(subset_query) if subset_query else df
    order = data[col].value_counts().index

    plt.figure(figsize=figsize)
    ax = sns.countplot(
        data=data,
        y=col if horizontal else None,
        x=None if horizontal else col,
        order=order,
        color=color
    )

    if show_counts:
        for p in ax.patches:
            count = int(p.get_width() if horizontal else p.get_height())
            pos_x = p.get_width() + 0.5 if horizontal else p.get_x() + p.get_width() / 2
            pos_y = p.get_y() + p.get_height() / 2 if horizontal else p.get_height() + 0.5
            if horizontal:
                ax.text(count + 0.5, p.get_y() + p.get_height()/2, str(count), va='center')
            else:
                ax.text(p.get_x() + p.get_width()/2, count + 0.5, str(count), ha='center')

    ax.set_title(title or f"Count of {col}")
    ax.set_xlabel("Count" if horizontal else col)
    ax.set_ylabel(col if horizontal else "Count")
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`

# Function to extract mod directory from the URL
def get_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def get_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def get_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to extract all TITLE occurrences from .mod content
def get_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all COMMENT sections from .mod content
def get_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def get_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


def get_use_ion(content):
    """
    Extracts the ion names used in the 'USEION' statements from NEURON mod file content.

    Parameters:
    - content (str): The content of the .mod file.

    Returns:
    - list: A list of ions used in 'USEION' statements, or None if none are found.
    """
    if pd.isna(content):  
        return None
    
    # Find all occurrences of USEION followed by an ion name
    matches = re.findall(r"USEION\s+(\w+)", content, re.MULTILINE)

    return matches if matches else None


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def get_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def get_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def get_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def get_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def get_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def get_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def get_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def get_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  


def get_net_receive(content):
    """
    Extracts all NET_RECEIVE block arguments from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted NET_RECEIVE arguments, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of NET_RECEIVE and extract arguments
    matches = re.findall(r"^\s*NET_RECEIVE\s*\(\s*([\w, ]+)\s*\)", content, re.MULTILINE)

    if not matches:
        return None

    net_receive_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return net_receive_vars if net_receive_vars else None

#todo: modify pipeline so that if get_include points to a file that file will be included in the content too
def get_include(content):
    """
    Extracts the filename in the INCLUDE statement from MOD file content, ignoring comments.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted INCLUDE filenames, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of INCLUDE, ensuring commented-out ones (with ':') are ignored
    matches = re.findall(r"^\s*INCLUDE\s+\"([^\"]+)\"", content, re.MULTILINE)

    return matches if matches else None


def get_point_process(content):
    """
    Extracts the POINT_PROCESS name from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        str or None: The extracted POINT_PROCESS name, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Extract the POINT_PROCESS name, ignoring comments
    match = re.search(r"^\s*POINT_PROCESS\s+([^\n:]+)", content, re.MULTILINE)

    return match.group(1).strip() if match else None


    
# Function to extract webpage heading
def get_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def get_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def get_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def get_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found






def has_electrode_or_clamp(mod_name, content):
    """
    Checks whether 'clamp' is present in the mod file name OR 
    'ELECTRODE_CURRENT' is present in the NEURON mod file content.

    Parameters:
    - mod_name (str): The name of the .mod file.
    - content (str): The content of the .mod file.

    Returns:
    - int: 1 if either 'clamp' is in the mod name OR 'ELECTRODE_CURRENT' is in the content, 0 otherwise.
    """
    if pd.isna(mod_name) and pd.isna(content):
        return None  # Return None if both are missing

    has_clamp = bool(re.search(r"clamp", str(mod_name), re.IGNORECASE)) if pd.notna(mod_name) else False
    has_electrode = bool(re.search(r"\bELECTRODE_CURRENT\b", str(content))) if pd.notna(content) else False

    return 1 if has_clamp or has_electrode else 0

def map_ion(value):
    value = value.lower()  # Normalize to lowercase

    # Define regex-based categorization rules
    patterns = [
        (r'ca.*i$', 'ca_i'),
        (r'ca.*o$', 'ca_o'),
        (r'cl.*i$', 'cl_i'),
        (r'cl.*o$', 'cl_o'),
        (r'k.*i$', 'k_i'),
        (r'k.*o$', 'k_o'),
        (r'na.*i$', 'na_i'),
        (r'na.*o$', 'na_o'),
        (r'hco3.*i$', 'other_i'),
        (r'hco3.*o$', 'other_o'),
        (r'mgi$', 'mg_i'),  
        (r'mgo$', 'mg_o'),  
        (r'^img$', 'i_mg'),  
        (r'^emg$', 'e_mg'),
        (r'^e.*ca', 'e_ca'),
        (r'^e.*k', 'e_k'),
        (r'^e.*na', 'e_na'),
        (r'^e.*mg', 'e_mg'),
        (r'^e.*', 'e_other'),
        (r'^i.*ca', 'i_cal'),
        (r'^i.*k', 'i_k'),
        (r'^i.*cl', 'i_cl'),
        (r'^i.*na$', 'i_na'),  # FIX: Ensure "ina" is classified as "i_na"
        (r'^i.*mg', 'i_mg'),
        (r'^i.*', 'i_other'),
        (r'.*i$', 'other_i'),  # General rule: Anything ending in "i" is "other_i"
        (r'.*o$', 'other_o')   # General rule: Anything ending in "o" is "other_o"
    ]
    # Apply the regex patterns
    for pattern, category in patterns:
        if re.search(pattern, value):
            return category

    return "unknown"  # Default category if no match is found

def count_states(df, column_name="state"):
    """Counts the number of states in each row of the specified column."""
    df["count_states"] = df[column_name].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df



def get_tau(param_dict):
    if not isinstance(param_dict, dict):
        return None, None  # Return separate None values for direct unpacking

    # Extract values where the key contains 'tau'
    tau_values = [v for k, v in param_dict.items() if 'tau' in k.lower()]
    
    # If no tau values found, return (None, None)
    if not tau_values:
        return None, None
    
    # Compute min and max
    return min(tau_values), max(tau_values)


def get_e(param_dict):
    if not isinstance(param_dict, dict):
        return [None, None]  # Handle cases where the value is not a dictionary

    #todo: modify the v pattern so it takes like 2 characters max
    # Regex pattern to match variations of reversal potential names
    pattern = re.compile(r"^(e|rev|v|shift).*", re.IGNORECASE)

    # Extract values where the key matches the pattern
    e_values = [v for k, v in param_dict.items() if pattern.match(k)]

    # If no values found, return [None, None]
    if not e_values:
        return [None, None]

    # Compute min and max reversal potential
    return min(e_values), max(e_values)



def has_mg(content):
    """
    Checks if 'mg' appears anywhere in the given content.

    Args:
        content (str): The text content to search.

    Returns:
        int: 1 if 'mg' is found, 0 otherwise.
    """
    if pd.isna(content):  # Handle missing content
        return 0

    return int(bool(re.search(r"mg", content, re.IGNORECASE)))  # Convert Boolean to int


def has_ih(content):

    if pd.isna(content):  # Handle missing content
        return 0

    return int(bool(re.search(r"ih", content, re.IGNORECASE)))  # Convert Boolean to int


# Import the Annotation Data

In [4]:
fp = "/gpfs/gibbs/project/mcdougal/imc33/mod-extract/data/model_db_annotations.xlsx"
ant_df_wide = pd.read_excel(fp).query("annotated == 'y' & ask_robert == 'n'")
vocab_df = pd.read_excel(fp, sheet_name="ref", usecols=['vocab_name','alias','type'])

/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed


In [5]:
vocab_df2 = (vocab_df
    .assign(type = lambda df: df["type"].fillna('neither'))
    .assign(
        vocab_name = lambda df: df["alias"].apply(
            lambda x: "Neither" if "Neither" in x else ("Exclude" if "Exclude" in x else x))))


In [6]:
vocab_df2["alias"].unique()

array(['Exclude - Not on Model DB', 'Exclude - Old Architecture',
       'Neither - Clamp', 'Neither - Utility', 'Neither - Non-Neural',
       'Neither - Artificial Cell', 'Neither - Accumulation Mechanism',
       'Neither  - Localized Reactions', 'Neither - Gap Junction',
       'I (Nonspecific)', 'I Ca (Ca Pump)',
       'I Ca (Calcium-Activiated Non-Specific)', 'I Ca (General)',
       'I Ca (L-Type HT)', 'I Ca (P/Q-Type)', 'I Ca (Persistent)',
       'I Ca (Q-Type)', 'I Ca (R-Type)', 'I Ca (SER)', 'I Ca (SOCC)',
       'I Ca (T-type LT)', 'I Cl (Ca-Activated ANO2)',
       'I Cl (Ca-Activated)', 'I Cl (General)', 'I Cl (Leak)',
       'I H (Hyperpolarization-Activated)', 'I K (AHP)',
       'I K (ATP-Sensitive)', 'I K (A-Type Slow)',
       'I K (A-Type Transient)',
       'I K (Ca-activated - General - BK, IK, SK, and I AHP currents)',
       'I K (Ca-Activated - Sk)', 'I K (Ca-activated Bk)', 'I K (CNQ1)',
       'I K (C-type)', 'I K (Delayed Rectifier)', 'I K (D-type)',
      

In [ ]:
a =set(vocab_df2['alias'])
b = set(pd.unique(ant_df_wide[['subtype_1', 'subtype_2', 'subtype_3', 'subtype_4']].values.ravel()))
EXCLUDED_VOCAB = sorted(a-b)
print("Excluding the following")
print(EXCLUDED_VOCAB)

In [ ]:
vocab_df3 = (vocab_df2
.query("alias not in @EXCLUDED_VOCAB")
.reset_index(drop=True)
)

In [ ]:
ant_df_long = (ant_df_wide.melt(
    id_vars=[col for col in ant_df_wide.columns if col not in ['subtype_1', 'subtype_2', 'subtype_3', 'subtype_4','subtype_5']],
    value_vars=['subtype_1', 'subtype_2', 'subtype_3', 'subtype_4','subtype_5'],
    var_name="subtype_num",
    value_name="original_subtype_label"
).dropna(subset=["original_subtype_label"])
.drop(columns=['raw_sha','count','url','mod_file','ask_robert','annotated','Robert reviewed','subtype_num'])
.sort_values('row_id'))

# Countplots

In [ ]:
# Loop through each unique type and plot
for t in ant_df_long["type"].dropna().unique():
    plot_countplot(
        df=ant_df_long,
        col='label',
        subset_query=f"type == '{t}'",
        title=f"Subtype Counts for: {t}",
        show_counts=True
    )


# Feature Engineering

In [ ]:
label_df = pd.read_csv("../data/label_df.csv").rename(columns={"original_subtype":"label"}).drop(columns=["subtype_id"])

In [ ]:
json_df = pd.read_json("../data/mod_files.json")

In [ ]:
json_df2 = (
    json_df
    .assign(
        exclude_error = lambda df: df["error_code"].notna().astype(int),
        exclude_x86 = lambda df: df["url"].apply(lambda x: 1 if isinstance(x, str) and "x86_64" in x else 0)
    )
    .merge(ant_df_long, on=["row_id", "file_hash"], how="inner")
    .assign(
        mod_name = lambda df: df["url"].apply(get_fname),
        suffix = lambda df: df["content"].apply(get_suffix),
        read_ion = lambda df: df["content"].apply(get_read_ion),
        read_ion2 = lambda df: df["read_ion"].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else []),
        write_ion = lambda df: df["content"].apply(get_write_ion),
        write_ion2 = lambda df: df["write_ion"].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else []),
        parameter = lambda df: df["content"].apply(get_parameter),
        state = lambda df: df["content"].apply(get_state),
        net_receive = lambda df: df["content"].apply(get_net_receive),
        point_process = lambda df: df["content"].apply(get_point_process),
        nonspecific_current = lambda df: df["content"].apply(get_nonspecific_current),

        # Discrete features
        states_count = lambda df: df["state"].apply(lambda x: len(x) if isinstance(x, list) else 0),

        # Binary features
        clamp_yn = lambda df: df.apply(lambda row: has_electrode_or_clamp(row["mod_name"], row["content"]), axis=1),
        suffix_yn = lambda df: df["suffix"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 
                                                  else (0 if pd.isna(x) or x == "none" else 1)),
        point_process_yn = lambda df: df["point_process"].apply(lambda x: 1 if pd.notna(x) else 0),
        net_receive_yn = lambda df: df["net_receive"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0)),
        i_nonspecific_yn = lambda df: df["nonspecific_current"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0)),
        not_ion_channel_yn = lambda df: ((df["suffix_yn"] == 0) & 
                                         df["read_ion"].isna() & 
                                         df["write_ion"].isna() & 
                                         df["nonspecific_current"].isna()).astype(int),
        not_receptor_yn = lambda df: ((df["point_process_yn"] == 0) & 
                                      (df["net_receive_yn"] == 0)).astype(int),
        has_mg_yn = lambda df: df["content"].apply(has_mg),
        has_ih_yn = lambda df: df["content"].apply(has_ih)
    ).merge(label_df, how="left", on="label")
)      

In [ ]:
json_df2

In [ ]:
json_df2["subtype_label"].value_counts()

In [ ]:
# Use MultiLabelBinarizer separately for read and write ions
mlb_read = MultiLabelBinarizer()
df_read_exploded = pd.DataFrame(mlb_read.fit_transform(df['read_ion2']), 
                                columns=[f'read_{col}_yn' for col in mlb_read.classes_])

mlb_write = MultiLabelBinarizer()
df_write_exploded = pd.DataFrame(mlb_write.fit_transform(df['write_ion2']), 
                                 columns=[f'write_{col}_yn' for col in mlb_write.classes_])

# Reset index before merging to avoid index mismatches
df = df.reset_index(drop=True)
df_read_exploded = df_read_exploded.reset_index(drop=True)
df_write_exploded = df_write_exploded.reset_index(drop=True)
df = df.drop(columns=['read_ion2', 'write_ion2']).join(df_read_exploded, rsuffix='_read_dup').join(df_write_exploded, rsuffix='_write_dup')

In [ ]:
df_pre = df.set_index('file_hash').filter(regex=r'(_yn|_count|new_label)$')

In [ ]:
df_pre.to_csv("preprocessed.csv")

# XGB model

In [ ]:
df_pre = pd.read_csv("preprocessed.csv", index_col='file_hash')

In [ ]:
# Load and split data
X = df_pre.drop(columns=['new_label'])  # Features
y = df_pre['new_label']  # Target

# Stratified 80/20 train-test split to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

# Merge train/test sets
df_train = X_train.copy()
df_train['new_label'] = y_train

df_test = X_test.copy()
df_test['new_label'] = y_test



In [ ]:
# Encode categorical labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  
y_test_encoded = label_encoder.transform(y_test)  

# Save the label encoder for later use
with open("label_encoder.pkl", "wb") as file:
    pickle.dump(label_encoder, file)

In [ ]:
# Define XGBoost model
xgb_model = xgb.XGBClassifier(
    objective="multi:softmax",
    num_class=len(label_encoder.classes_),
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

In [ ]:
# Perform 5-Fold Cross-Validation on the Training Set
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train, y_train_encoded, cv=kf, scoring='accuracy')

print(f"Cross-validation scores: {cv_scores}")
print(f"Mean Accuracy (CV): {cv_scores.mean():.4f}")

In [ ]:
# Train on Full Training Set
xgb_model.fit(X_train, y_train_encoded, verbose=True)

# Save the trained XGBoost model
with open("xgb_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Predict on test data
y_test_pred_enc = xgb_model.predict(X_test)  # Get encoded predictions

# Convert predictions back to original labels
y_test_pred = label_encoder.inverse_transform(y_test_pred_enc)

# Accuracy on test data
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate classification report with meaningful labels
print("\nClassification Report on Test Set:")
print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))


In [ ]:

# Predict on test data
y_test_pred_enc = xgb_model.predict(X_test)  # Get encoded predictions

# Convert predictions back to original labels
y_test_pred = label_encoder.inverse_transform(y_test_pred_enc)

# Accuracy on test data
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate classification report as a dictionary
report = classification_report(y_test, y_test_pred, target_names=label_encoder.classes_, output_dict=True)

# Convert to DataFrame
df_report = pd.DataFrame(report).transpose()

# Filter only worst-performing subtypes (F1-score == 0.00)
df_worst = df_report[df_report["f1-score"] == 0.00]

# Drop support column for better readability
df_worst = df_worst.drop(columns=["support"])

df_worst

In [ ]:
df_test

# Plots

In [ ]:
df_train.shape

In [ ]:
df_test.shape

In [ ]:
subtypes = df_train[df_train["new_label"] != "z_neither"]

In [ ]:
# Sort the DataFrame by 'new_label' in alphabetical order
subtypes = subtypes.sort_values(by='new_label')

# Create a count plot with different colors
plt.figure(figsize=(8, 5))
ax = sns.countplot(y='new_label', data=subtypes, order=subtypes['new_label'].unique())

# Ensure only whole numbers appear on the x-axis
max_count = subtypes['new_label'].value_counts().max()  # Get max count for setting x-ticks
ax.set_xticks(np.arange(0, max_count + 1, 1))  # Set integer x-ticks

# Add a title
plt.title('Training Set')

# Show the plot
plt.show()

In [ ]:
subtypes = df_test[df_test["new_label"] != "z_neither"]

In [ ]:
# Sort the DataFrame by 'new_label' in alphabetical order
subtypes = subtypes.sort_values(by='new_label')

# Create a count plot with different colors
plt.figure(figsize=(8, 5))
ax = sns.countplot(y='new_label', data=subtypes, order=subtypes['new_label'].unique())

# Ensure only whole numbers appear on the x-axis
max_count = subtypes['new_label'].value_counts().max()  # Get max count for setting x-ticks
ax.set_xticks(np.arange(0, max_count + 1, 1))  # Set integer x-ticks

# Add a title
plt.title('Test Set')

# Show the plot
plt.show()

In [ ]:


# Plot feature importances
plt.figure(figsize=(10, 6))
xgb.plot_importance(xgb_model, max_num_features=10)  # Show top 10 features
plt.title("Feature Importance in XGBoost")
plt.show()


In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

# Ensure predictions are in encoded numeric form
y_test_pred_encoded = label_encoder.transform(y_test_pred)  # Convert back to numeric if necessary

# Generate confusion matrix
conf_matrix = confusion_matrix(y_test_encoded, y_test_pred_encoded)

# Convert to DataFrame for better visualization
conf_df = pd.DataFrame(conf_matrix, 
                       index=label_encoder.classes_, 
                       columns=label_encoder.classes_)


In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(conf_df, annot=True, fmt="d", cmap="Blues", linewidths=0.5)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix Heatmap")
plt.show()


In [ ]:


# Create an explainer
explainer = shap.Explainer(xgb_model, X_test)

# Compute SHAP values
shap_values = explainer(X_test)


In [ ]:
#H-Current
plt.title("H-Current")
shap.plots.beeswarm(shap_values[:,:,3])

In [ ]:
plt.title("K A-type (2 gates)")
shap.plots.beeswarm(shap_values[:,:,4])

In [ ]:
plt.title("K Delayed Rectifier (1 gate)")
shap.plots.beeswarm(shap_values[:,:,6])

In [ ]:
plt.title("K Ca Channel")
shap.plots.beeswarm(shap_values[:,:,5])

In [ ]:
plt.title("Neither")
shap.plots.beeswarm(shap_values[:,:,-1])

# Compare to GPT

In [ ]:
df_gpt = pd.read_csv("../data/mod_files_merged 3.csv")

In [ ]:
df_gpt_subset = df_gpt[df_gpt["hash"].isin(X_test.index)].copy()

In [ ]:
# List of column names to process
current_cols = ["currents_at_least_1", "currents_at_least_2", "currents_at_least_3"]

# Convert string lists into actual Python lists for all columns
for col in current_cols:
    df_gpt_subset[col] = df_gpt_subset[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode all columns, drop NaNs, and extract unique values
unique_currents = pd.concat([df_gpt_subset[col].explode() for col in current_cols]).dropna().unique()

# Convert to a sorted list (optional)
unique_currents_list = sorted(unique_currents)

# Print unique current names
print(unique_currents_list)


In [ ]:
columns_to_update = ["currents_at_least_1", "currents_at_least_2", "currents_at_least_3"]

# Apply the function to each element in the specified columns
for col in columns_to_update:
    df_gpt_subset[col] = df_gpt_subset[col].apply(lambda x: [map_broad_label(val) for val in x] if isinstance(x, list) else x)


In [ ]:
View(df_gpt_subset[df_gpt_subset["currents_at_least_1"].apply(lambda x: "check" in x)])

In [ ]:
# Ensure `currents_at_least_3` is properly formatted as a list
df_gpt_subset["currents_at_least_3"] = df_gpt_subset["currents_at_least_3"].apply(
    lambda x: x if isinstance(x, list) else [x] if pd.notna(x) else []
)

# 🔹 Step 1: Convert to Wide Format (One column per element in the list)
max_len = df_gpt_subset["currents_at_least_3"].apply(len).max()  # Max number of elements in a row

# Create new columns (current_1, current_2, ..., current_n)
df_wide = df_gpt_subset.copy()
df_wide[[f"current_{i+1}" for i in range(max_len)]] = pd.DataFrame(df_wide["currents_at_least_3"].tolist(), index=df_wide.index)

# 🔹 Step 2: Convert to Long Format (One current per row)
df_long = df_wide.melt(id_vars=["hash"], value_vars=[f"current_{i+1}" for i in range(max_len)],
                        var_name="current_position", value_name="gpt_pred").dropna()

# Drop the helper column and reset index
df_long = df_long.drop(columns=["current_position"]).reset_index(drop=True)

In [ ]:
#df_long[df_long["gpt_pred"]!="check"]

In [ ]:
df_xgb_pred = pd.DataFrame({"hash": X_test.index, "xgb_pred": y_test_pred, "label":y_test}).set_index("hash")  # XGB predictions
df_xgb_pred = df_xgb_pred.reset_index()

In [ ]:
df_both = df_xgb_pred.merge(df_long, how="left").fillna("z_neither")

In [ ]:
df_both

In [ ]:


# Extract ground truth labels and GPT predictions
y_true = df_both["label"]  # True labels
y_pred = df_both["gpt_pred"]  # GPT model predictions

# Compute accuracy
gpt_accuracy = accuracy_score(y_true, y_pred)
print(f"GPT Model Accuracy: {gpt_accuracy:.4f}")

# Generate classification report
print("Classification Report for GPT Model:")
print(classification_report(y_true, y_pred))

# Compute confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))

# Convert confusion matrix to DataFrame for visualization
conf_df = pd.DataFrame(conf_matrix, index=np.unique(y_true), columns=np.unique(y_true))

# 🔹 Plot Confusion Matrix Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(conf_df, annot=True, fmt="d", cmap="Blues", linewidths=0.5)
plt.xlabel("Predicted Label (GPT Model)")
plt.ylabel("Actual Label (Ground Truth)")
plt.title("Confusion Matrix for GPT Model")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# Extract ground truth labels and predictions
y_true = df_both["label"]  # Ground truth
y_xgb = df_both["xgb_pred"]  # XGB model predictions
y_gpt = df_both["gpt_pred"]  # GPT model predictions

# Generate classification reports as dictionaries
xgb_report = classification_report(y_true, y_xgb, output_dict=True)
gpt_report = classification_report(y_true, y_gpt, output_dict=True)

# Extract F1-scores for each class and create a DataFrame
f1_scores = pd.DataFrame({
    "Class": list(xgb_report.keys())[:-3],  # Exclude "accuracy", "macro avg", "weighted avg"
    
    "XGB F1": [xgb_report[label]["f1-score"] for label in xgb_report.keys() if label not in ["accuracy", "macro avg", "weighted avg"]],
    "GPT F1": [gpt_report[label]["f1-score"] for label in gpt_report.keys() if label not in ["accuracy", "macro avg", "weighted avg"]],
})


In [ ]:
f1_scores.set_index('Class')

# Flagging Issues

In [ ]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)
#row_id 99 - need to fix use_ion

# Questions

In [ ]:
#todo: need to get INCLUDE for the notes table
#todo: should we collapse low-frequency labels
#todo: how do we process the SUFFIX, a lot of times the SUFFIX actually is the actual labeL?
#todo: do we include the name
#df["label"].value_counts()


In [ ]:
#REsol;ved Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? (don't capture stuff commented out)
#what's the best way to capture 


In [ ]:
View(df.loc[481,["url","mod_state"]],50)

In [ ]:
import re





In [ ]:
! git add .
! git commit -m "added broad level"
! git push 